# Generate 7 Publication-Quality Figures from Experiment Outputs

This notebook reproduces the evaluation pipeline that generates **7 publication-quality matplotlib figures** (300 DPI PNGs) from experiment outputs for the Signed Spectral FIGS paper:

1. **Pipeline diagram** - Method overview schematic
2. **Critical difference diagram** - Friedman + Nemenyi statistical test
3. **Grouped accuracy bars** - 8 methods x 7 datasets comparison
4. **Arity-accuracy Pareto scatter** - Interpretability tradeoff
5. **Forest plot** - Signed vs unsigned Hedges' g effect sizes
6. **Module recovery bars** - Synergistic pair Jaccard scores
7. **Scaling analysis** - Log-log computation time with power-law fits

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# scikit-posthocs, loguru — NOT on Colab, always install
_pip('scikit-posthocs==0.11.0')
_pip('loguru==0.7.3')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    # scipy 1.16.3 needs Python >=3.11 (Colab uses 3.12); fall back for 3.10
    _scipy = 'scipy==1.16.3' if sys.version_info >= (3, 11) else 'scipy>=1.13,<1.16'
    _pip('numpy==2.0.2', 'pandas==2.2.2', _scipy, 'matplotlib==3.10.0')


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


## Imports

In [2]:
import json
import sys
import os
import math
import gc
from pathlib import Path
from collections import defaultdict

import numpy as np

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from scipy import stats
import scikit_posthocs as sp
from loguru import logger

# Setup logging for notebook
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

1

## Load Demo Data

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-7dffcf-balance-guided-oblique-trees-signed-spec/main/evaluation_iter7_generate_7_publ/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
print(f"Loaded data with keys: {list(data.keys())}")
print(f"FIGS data entries: {len(data['figs_data'])}")
print(f"Baseline data entries: {len(data['base_data'])}")

Loaded data with keys: ['figs_data', 'base_data', 'exp3_per_variant_results', 'exp1_recov_per_variant', 'exp2_frust_per_dataset_results']
FIGS data entries: 35
Baseline data entries: 21


## Configuration

In [5]:
# ── Tunable parameters ──
DPI = 150  # Original: 300 — reduced for faster demo rendering
N_BOOTSTRAP = 1000  # Original: 10000 — reduced for speed

# ── Constants (from original eval.py) ──
CLASSIFICATION_DATASETS = [
    'adult', 'electricity', 'credit', 'eye_movements',
    'higgs_small', 'miniboone', 'jannis',
]
DATASET_NFEATURES = {
    'adult': 6, 'electricity': 7, 'credit': 10,
    'eye_movements': 20, 'higgs_small': 24,
    'miniboone': 50, 'jannis': 54,
}

FIGS_METHODS = [
    'axis_aligned', 'random_oblique', 'hard_threshold',
    'unsigned_spectral', 'signed_spectral',
]
BASELINE_METHODS = ['ebm', 'random_forest', 'linear']
ALL_METHODS = FIGS_METHODS + BASELINE_METHODS

METHOD_COLORS = {
    'axis_aligned': '#BBBBBB', 'random_oblique': '#4477AA',
    'hard_threshold': '#66CCEE', 'unsigned_spectral': '#228833',
    'signed_spectral': '#EE6677', 'ebm': '#AA3377',
    'random_forest': '#CCBB44', 'linear': '#DDDDDD',
}
METHOD_LABELS = {
    'axis_aligned': 'FIGS (Axis)', 'random_oblique': 'FIGS (Rand-Obl.)',
    'hard_threshold': 'FIGS (Hard-Thr.)', 'unsigned_spectral': 'FIGS (Uns-Spec.)',
    'signed_spectral': 'FIGS (Sgn-Spec.)', 'ebm': 'EBM',
    'random_forest': 'Random Forest', 'linear': 'Logistic/Ridge',
}
METHOD_MARKERS = {
    'axis_aligned': 'o', 'random_oblique': 's', 'hard_threshold': 'D',
    'unsigned_spectral': '^', 'signed_spectral': 'p',
}
NEMENYI_Q = {
    2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728,
    6: 2.850, 7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164,
}
SYNTH_VARIANTS_RECOVERY = [
    'easy_2mod_xor', 'medium_4mod_mixed', 'hard_4mod_unequal',
    'overlapping_modules', 'highdim_8mod',
]
SYNTH_VARIANT_LABELS = {
    'easy_2mod_xor': 'Easy (2-mod)', 'medium_4mod_mixed': 'Medium (4-mod)',
    'hard_4mod_unequal': 'Hard (unequal)', 'overlapping_modules': 'Overlapping',
    'highdim_8mod': 'High-dim (8-mod)', 'no_structure_control': 'No Structure',
}
ALL_SYNTH_VARIANTS = [
    'easy_2mod_xor', 'medium_4mod_mixed', 'hard_4mod_unequal',
    'overlapping_modules', 'no_structure_control', 'highdim_8mod',
]

# Academic style
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'Times'],
    'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 12,
    'legend.fontsize': 9, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'figure.dpi': DPI, 'savefig.dpi': DPI,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.1,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

## Pre-process Data

Convert JSON data to the dictionary format used by figure functions.

In [6]:
# Reconstruct figs_data: {(dataset, method): {...}}
figs_data = {}
for key, val in data['figs_data'].items():
    ds, m = key.split('|')
    figs_data[(ds, m)] = val

# Reconstruct base_data: {(dataset, method): {...}}
base_data = {}
for key, val in data['base_data'].items():
    ds, m = key.split('|')
    base_data[(ds, m)] = val

# Synthetic experiment data for forest plot
exp3_meta = {'per_variant_results': data['exp3_per_variant_results']}

# Module recovery data
exp1_recov_meta = {'per_variant': data['exp1_recov_per_variant']}

# Scaling data
exp2_frust_meta = {'per_dataset_results': data['exp2_frust_per_dataset_results']}

print(f"FIGS data: {len(figs_data)} (dataset, method) pairs")
print(f"Baseline data: {len(base_data)} (dataset, method) pairs")
print(f"Synthetic variants: {len(exp3_meta['per_variant_results'])}")
print(f"Recovery variants: {len(exp1_recov_meta['per_variant'])}")
print(f"Scaling datasets: {len(exp2_frust_meta['per_dataset_results'])}")

FIGS data: 35 (dataset, method) pairs
Baseline data: 21 (dataset, method) pairs
Synthetic variants: 6
Recovery variants: 5
Scaling datasets: 14


## Utility Functions

In [7]:
def sanitize_number(v) -> float:
    if v is None:
        return 0.0
    v = float(v)
    if np.isnan(v) or np.isinf(v):
        return 0.0
    return v

def hedges_g(group1: list, group2: list) -> tuple:
    """Compute Hedges' g and bootstrap 95% CI."""
    a1 = np.array(group1, dtype=float)
    a2 = np.array(group2, dtype=float)
    n1, n2 = len(a1), len(a2)
    if n1 < 2 or n2 < 2:
        return 0.0, (-1.0, 1.0)
    m1, m2 = a1.mean(), a2.mean()
    s1, s2 = a1.std(ddof=1), a2.std(ddof=1)
    pooled_sd = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    if pooled_sd < 1e-12:
        return 0.0, (-0.5, 0.5)
    J = 1 - 3 / (4 * (n1 + n2 - 2) - 1)
    g = (m1 - m2) / pooled_sd * J
    rng = np.random.default_rng(42)
    boot_gs = np.empty(N_BOOTSTRAP)
    for b in range(N_BOOTSTRAP):
        idx1 = rng.integers(0, n1, size=n1)
        idx2 = rng.integers(0, n2, size=n2)
        b1, b2 = a1[idx1], a2[idx2]
        bp = np.sqrt(((n1-1)*b1.std(ddof=1)**2 + (n2-1)*b2.std(ddof=1)**2) / (n1+n2-2))
        boot_gs[b] = (b1.mean() - b2.mean()) / bp * J if bp > 1e-12 else 0.0
    return float(g), (float(np.percentile(boot_gs, 2.5)), float(np.percentile(boot_gs, 97.5)))

def find_cd_cliques(sorted_ranks: list, cd: float) -> list:
    n = len(sorted_ranks)
    cliques = []
    for i in range(n):
        for j in range(n - 1, i, -1):
            if sorted_ranks[j] - sorted_ranks[i] <= cd:
                cliques.append((i, j))
                break
    final = []
    for c in cliques:
        is_subset = any(o[0] <= c[0] and o[1] >= c[1] and o != c for o in cliques)
        if not is_subset and c[0] != c[1]:
            final.append(c)
    return final

print("Utility functions defined.")

Utility functions defined.


## Figure 1: Method Pipeline Diagram

In [8]:
fig, ax = plt.subplots(1, 1, figsize=(10, 3))
ax.set_xlim(-0.2, 11.0)
ax.set_ylim(-1.2, 2.2)
ax.axis('off')

stages = [
    (0.0, 1.8, 'Raw Features', '$X_1 \ldots X_d$', '#F0F0F0'),
    (2.5, 1.8, 'Pairwise\nCo-Information', 'CoI matrix', '#E8D5E0'),
    (5.0, 1.8, 'Signed\nGraph', '+/− edges', '#D5E0E8'),
    (7.5, 1.8, 'Spectral\nClustering', 'SPONGE', '#D5E8D5'),
    (10.0, 1.4, 'FIGS\nEnsemble', 'oblique splits', '#F5E6D0'),
]
for x, w, title, sub, color in stages:
    h = 1.2; y = 0.2
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.12",
        facecolor=color, edgecolor='#333333', linewidth=1.3)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + 0.1, title, ha='center', va='center', fontsize=9, fontweight='bold')
    ax.text(x + w/2, y + 0.15, sub, ha='center', va='center', fontsize=7, fontstyle='italic', color='#555555')

arrow_kw = dict(arrowstyle="Simple,tail_width=3,head_width=10,head_length=6", color='#555555')
for x1, x2 in [(1.8, 2.5), (4.3, 5.0), (6.8, 7.5), (9.3, 10.0)]:
    ax.add_patch(FancyArrowPatch((x1+0.05, 0.8), (x2-0.05, 0.8), **arrow_kw))

ax.set_title('Signed Spectral FIGS Pipeline', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()
print("Figure 1: Pipeline diagram generated.")

Figure 1: Pipeline diagram generated.


## Figure 2: Critical Difference Diagram

Friedman test + Nemenyi post-hoc to compare 8 methods across 7 datasets.

In [9]:
n_ds = len(CLASSIFICATION_DATASETS)
n_m = len(ALL_METHODS)

matrix = np.full((n_ds, n_m), 0.5)
for i, ds in enumerate(CLASSIFICATION_DATASETS):
    for j, method in enumerate(ALL_METHODS):
        key = (ds, method)
        if key in figs_data:
            matrix[i, j] = figs_data[key]['ba_mean']
        elif key in base_data:
            matrix[i, j] = base_data[key]['ba_mean']

fstat, fpval = stats.friedmanchisquare(*[matrix[:, j] for j in range(n_m)])
print(f"Friedman chi2={fstat:.3f}, p={fpval:.6f}")

ranks = np.zeros_like(matrix)
for i in range(n_ds):
    ranks[i, :] = stats.rankdata(-matrix[i, :])
avg_ranks = ranks.mean(axis=0)

order = np.argsort(avg_ranks)
s_methods = [ALL_METHODS[i] for i in order]
s_ranks = avg_ranks[order].tolist()

q_alpha = NEMENYI_Q.get(n_m, 3.031)
cd = q_alpha * np.sqrt(n_m * (n_m + 1) / (6 * n_ds))
cliques = find_cd_cliques(s_ranks, cd)
print(f"CD = {cd:.3f}, Cliques: {len(cliques)}")

# Draw
fig, ax = plt.subplots(figsize=(8, 4))
y_axis = 0.0
ax.set_xlim(0.5, n_m + 0.5)
half = n_m // 2

ax.hlines(y_axis, 1, n_m, color='black', linewidth=1)
for t in range(1, n_m + 1):
    ax.vlines(t, y_axis - 0.08, y_axis + 0.08, color='black', linewidth=1)
    ax.text(t, y_axis + 0.15, str(t), ha='center', va='bottom', fontsize=8)

y_cd = y_axis + 0.55
ax.hlines(y_cd, 1, 1 + cd, color='#333333', linewidth=2)
ax.vlines(1, y_cd - 0.06, y_cd + 0.06, color='#333333', linewidth=2)
ax.vlines(1 + cd, y_cd - 0.06, y_cd + 0.06, color='#333333', linewidth=2)
ax.text(1 + cd/2, y_cd + 0.1, f'CD = {cd:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

for idx in range(half):
    m = s_methods[idx]; r = s_ranks[idx]
    y_label = y_axis + 0.6 + idx * 0.5
    color = METHOD_COLORS[m]
    weight = 'bold' if m == 'signed_spectral' else 'normal'
    ax.plot(r, y_axis, 'o', color=color, markersize=7, markeredgecolor='black', markeredgewidth=0.5, zorder=5)
    ax.plot([r, r], [y_axis + 0.08, y_label - 0.08], color='#888888', linewidth=0.6)
    ax.plot([r, 0.8], [y_label, y_label], color='#888888', linewidth=0.6)
    ax.text(0.75, y_label, f'{METHOD_LABELS[m]} ({r:.2f})', ha='right', va='center', fontsize=8, color=color, fontweight=weight)

for idx_offset, idx in enumerate(range(half, n_m)):
    m = s_methods[idx]; r = s_ranks[idx]
    y_label = y_axis + 0.6 + idx_offset * 0.5
    color = METHOD_COLORS[m]
    weight = 'bold' if m == 'signed_spectral' else 'normal'
    ax.plot(r, y_axis, 'o', color=color, markersize=7, markeredgecolor='black', markeredgewidth=0.5, zorder=5)
    ax.plot([r, r], [y_axis + 0.08, y_label - 0.08], color='#888888', linewidth=0.6)
    ax.plot([r, n_m + 0.2], [y_label, y_label], color='#888888', linewidth=0.6)
    ax.text(n_m + 0.25, y_label, f'{METHOD_LABELS[m]} ({r:.2f})', ha='left', va='center', fontsize=8, color=color, fontweight=weight)

for ci, (start, end) in enumerate(cliques):
    y_bar = y_axis - 0.3 - ci * 0.22
    ax.hlines(y_bar, s_ranks[start], s_ranks[end], color='#333333', linewidth=3.0)

ax.set_ylim(y_axis - 0.3 - len(cliques)*0.22 - 0.3, y_axis + 0.6 + max(half, n_m-half)*0.5 + 0.8)
ax.axis('off')
sig_str = 'significant' if fpval < 0.05 else 'not significant'
ax.set_title(f'Critical Difference Diagram\n(Friedman χ²={fstat:.1f}, p={fpval:.4f}, {sig_str})', fontsize=11, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

Friedman chi2=37.095, p=0.000004
CD = 3.969, Cliques: 2


## Figure 3: Accuracy Comparison (Grouped Bar Chart)

In [10]:
fig, ax = plt.subplots(figsize=(12, 5))
n_ds = len(CLASSIFICATION_DATASETS)
n_m = len(ALL_METHODS)
bar_w = 0.1
x_base = np.arange(n_ds)

for j, method in enumerate(ALL_METHODS):
    means, stds = [], []
    for ds in CLASSIFICATION_DATASETS:
        key = (ds, method)
        if key in figs_data:
            means.append(figs_data[key]['ba_mean']); stds.append(figs_data[key]['ba_std'])
        elif key in base_data:
            means.append(base_data[key]['ba_mean']); stds.append(base_data[key]['ba_std'])
        else:
            means.append(0.5); stds.append(0.0)
    offset = (j - n_m/2 + 0.5) * bar_w
    edge_kw = dict(edgecolor='black', linewidth=1.2) if method == 'signed_spectral' else dict(edgecolor='none')
    ax.bar(x_base + offset, means, bar_w, yerr=stds, color=METHOD_COLORS[method],
           label=METHOD_LABELS[method], capsize=2, error_kw={'linewidth': 0.7}, **edge_kw)

xlabels = [f"{ds}\n(d={DATASET_NFEATURES[ds]})" for ds in CLASSIFICATION_DATASETS]
ax.set_xticks(x_base); ax.set_xticklabels(xlabels, fontsize=8)
ax.set_ylabel('Balanced Accuracy'); ax.set_xlabel('Dataset (ordered by dimensionality)')
ax.set_title('Method Comparison Across Classification Datasets', fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7, frameon=True)
ax.set_ylim(0.4, 1.0)
ax.axhline(y=0.5, color='grey', linestyle=':', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.show()

## Figure 4: Arity-Accuracy Pareto Scatter

In [11]:
fig, ax = plt.subplots(figsize=(7, 5))
min_nf, max_nf = 6, 54
min_sz, max_sz = 30, 150
all_pts = []
for method in FIGS_METHODS:
    for ds in CLASSIFICATION_DATASETS:
        key = (ds, method)
        if key in figs_data:
            d = figs_data[key]
            all_pts.append({'arity': d['arity_mean'], 'accuracy': d['ba_mean'],
                           'method': method, 'dataset': ds, 'n_features': DATASET_NFEATURES[ds]})

for method in FIGS_METHODS:
    pts = [p for p in all_pts if p['method'] == method]
    if not pts: continue
    x = [p['arity'] for p in pts]; y = [p['accuracy'] for p in pts]
    sizes = [min_sz + (p['n_features']-min_nf)/max(1,max_nf-min_nf)*(max_sz-min_sz) for p in pts]
    ax.scatter(x, y, s=sizes, c=METHOD_COLORS[method], marker=METHOD_MARKERS[method],
               label=METHOD_LABELS[method], alpha=0.8, edgecolors='black', linewidths=0.5, zorder=3)

# Pareto frontier
frontier = []
if all_pts:
    sorted_by_arity = sorted(all_pts, key=lambda p: (p['arity'], -p['accuracy']))
    best_acc = -1
    for p in sorted_by_arity:
        if p['accuracy'] > best_acc:
            frontier.append((p['arity'], p['accuracy'])); best_acc = p['accuracy']
    if len(frontier) > 1:
        fx, fy = zip(*frontier)
        ax.step(list(fx), list(fy), where='post', color='#333333', linestyle='--',
                linewidth=1.5, alpha=0.5, label='Pareto frontier', zorder=2)

ax.set_xlabel('Mean Split Arity'); ax.set_ylabel('Mean Balanced Accuracy')
ax.set_title('Accuracy–Interpretability Tradeoff', fontweight='bold')
handles, labels = ax.get_legend_handles_labels()
for nf_val, lbl in [(6, 'd=6'), (30, 'd=30'), (54, 'd=54')]:
    sz = min_sz + (nf_val-min_nf)/max(1,max_nf-min_nf)*(max_sz-min_sz)
    handles.append(ax.scatter([], [], s=sz, c='grey', alpha=0.5, edgecolors='black', linewidths=0.5))
    labels.append(lbl)
ax.legend(handles, labels, fontsize=7, frameon=True, loc='lower right', ncol=2)
plt.tight_layout()
plt.show()

## Figure 5: Signed vs Unsigned Forest Plot (Hedges' g)

In [12]:
rows = []
hedges_g_results = {}

for ds in CLASSIFICATION_DATASETS:
    key_u = (ds, 'unsigned_spectral'); key_s = (ds, 'signed_spectral')
    if key_u in figs_data and key_s in figs_data:
        uf = figs_data[key_u]['per_fold_ba']; sf = figs_data[key_s]['per_fold_ba']
        if len(uf) >= 2 and len(sf) >= 2:
            g, (ci_lo, ci_hi) = hedges_g(uf, sf)
            rows.append((ds, g, ci_lo, ci_hi, 'Real'))
            hedges_g_results[ds] = float(g)

pvr = exp3_meta.get('per_variant_results', {})
for var in ALL_SYNTH_VARIANTS:
    if var not in pvr: continue
    methods = pvr[var].get('methods', {})
    u_folds = [f['balanced_accuracy'] for f in methods.get('unsigned_spectral', {}).get('best_folds', [])]
    s_folds = [f['balanced_accuracy'] for f in methods.get('signed_spectral', {}).get('best_folds', [])]
    if len(u_folds) >= 2 and len(s_folds) >= 2:
        g, (ci_lo, ci_hi) = hedges_g(u_folds, s_folds)
        label = SYNTH_VARIANT_LABELS.get(var, var)
        rows.append((label, g, ci_lo, ci_hi, 'Synthetic'))
        hedges_g_results[var] = float(g)

real_rows = [r for r in rows if r[4] == 'Real']
synth_rows = [r for r in rows if r[4] == 'Synthetic']
n_rows = len(rows)
fig_h = max(4, 0.55 * n_rows + 2.5)
fig, ax = plt.subplots(figsize=(8, fig_h))

y_positions, y_labels = [], []
total_height = len(real_rows) + len(synth_rows) + 2
y = total_height

def _draw_row(ax, y_val, g_val, ci_lo_val, ci_hi_val):
    color = '#EE6677' if g_val < 0 else '#4477AA'
    ci_w = max(abs(ci_hi_val - ci_lo_val), 0.01)
    sq_sz = max(4, min(12, 20 / ci_w))
    ax.plot([ci_lo_val, ci_hi_val], [y_val, y_val], color=color, linewidth=1.5, zorder=2)
    ax.plot(g_val, y_val, 's', color=color, markersize=sq_sz, markeredgecolor='black', markeredgewidth=0.5, zorder=3)

if real_rows:
    ax.text(0.01, y + 0.2, 'Real Datasets', fontweight='bold', fontsize=9, ha='left', va='bottom', transform=ax.get_yaxis_transform())
for name, g, ci_lo, ci_hi, _ in real_rows:
    y -= 1; y_positions.append(y); y_labels.append(name)
    _draw_row(ax, y, g, ci_lo, ci_hi)

if real_rows and synth_rows:
    y -= 0.5
    ax.axhline(y=y, color='grey', linestyle='--', linewidth=0.5, alpha=0.5, xmin=0.05, xmax=0.95)
    y -= 0.3
    ax.text(0.01, y + 0.2, 'Synthetic Variants', fontweight='bold', fontsize=9, ha='left', va='bottom', transform=ax.get_yaxis_transform())

for name, g, ci_lo, ci_hi, _ in synth_rows:
    y -= 1; y_positions.append(y); y_labels.append(name)
    _draw_row(ax, y, g, ci_lo, ci_hi)

ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8, alpha=0.7, zorder=1)
ax.set_yticks(y_positions); ax.set_yticklabels(y_labels, fontsize=9)
ax.set_xlabel("Hedges' g  (positive → unsigned better, negative \u2192 signed better)", fontsize=10)
ax.set_title('Signed vs Unsigned Spectral: Effect Sizes', fontweight='bold')
ax.annotate('← Signed better', xy=(0.02, 0.02), xycoords='axes fraction', fontsize=7, color='#EE6677', fontstyle='italic')
ax.annotate('Unsigned better →', xy=(0.98, 0.02), xycoords='axes fraction', fontsize=7, color='#4477AA', ha='right', fontstyle='italic')
ax.set_ylim(y - 0.5, total_height + 0.5)
plt.tight_layout()
plt.show()

## Figure 6: Synthetic Module Recovery Bars

In [13]:
per_variant = exp1_recov_meta.get('per_variant', {})
recovery_methods = ['sponge_oracle_k', 'sponge_auto_k', 'unsigned_spectral', 'hard_threshold']
recovery_colors = {'sponge_oracle_k': '#EE6677', 'sponge_auto_k': '#CC3355',
                   'unsigned_spectral': '#228833', 'hard_threshold': '#66CCEE'}
recovery_labels = {'sponge_oracle_k': 'SPONGE (Oracle k)', 'sponge_auto_k': 'SPONGE (Auto k)',
                   'unsigned_spectral': 'Unsigned Spectral', 'hard_threshold': 'Hard Threshold'}

fig, ax = plt.subplots(figsize=(9, 4.5))
n_var = len(SYNTH_VARIANTS_RECOVERY); n_meth = len(recovery_methods)
bar_w = 0.18; x_base = np.arange(n_var)

for j, method in enumerate(recovery_methods):
    jaccards = []
    for var in SYNTH_VARIANTS_RECOVERY:
        if var in per_variant:
            mdict = per_variant[var].get('methods', {})
            jac = mdict.get(method, {}).get('synergistic_pair_jaccard', 0.0)
            jaccards.append(float(jac) if jac is not None else 0.0)
        else:
            jaccards.append(0.0)
    offset = (j - n_meth/2 + 0.5) * bar_w
    ax.bar(x_base + offset, jaccards, bar_w, color=recovery_colors[method],
           label=recovery_labels[method], edgecolor='white', linewidth=0.5)

first_rp = True
for vi, var in enumerate(SYNTH_VARIANTS_RECOVERY):
    if var in per_variant:
        rp_jac = per_variant[var].get('methods', {}).get('random_partition', {}).get('synergistic_pair_jaccard')
        if rp_jac is not None:
            ax.plot(vi, float(rp_jac), 'x', color='grey', markersize=8, markeredgewidth=2, zorder=5,
                    label='Random Partition' if first_rp else None)
            first_rp = False

ax.axhline(y=0.8, color='#333333', linestyle='--', linewidth=1, alpha=0.5, label='Success threshold (0.8)')
xlabels = [SYNTH_VARIANT_LABELS.get(v, v) for v in SYNTH_VARIANTS_RECOVERY]
ax.set_xticks(x_base); ax.set_xticklabels(xlabels, fontsize=9)
ax.set_ylabel('Synergistic Pair Jaccard'); ax.set_xlabel('Synthetic Variant')
ax.set_title('Module Recovery: Synergistic Pair Jaccard', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend(fontsize=7, frameon=True, loc='upper right', ncol=2)
plt.tight_layout()
plt.show()

## Figure 7: Scaling Analysis (Log-Log with Power-Law Fits)

In [14]:
pdr = exp2_frust_meta.get('per_dataset_results', {})
data_pts = []
for ds, info in pdr.items():
    nf = info.get('n_features')
    coi_time = info.get('coi_computation', {}).get('time_s', 0)
    sponge_time = info.get('signed_spectral_sponge', {}).get('time_s', 0)
    if nf and nf > 0 and coi_time > 0:
        data_pts.append({'dataset': ds, 'n_features': nf, 'coi_time': coi_time,
                        'sponge_time': sponge_time, 'total_time': coi_time + sponge_time})

fig, ax = plt.subplots(figsize=(7, 5))
nf_arr = np.array([p['n_features'] for p in data_pts])
coi_arr = np.array([p['coi_time'] for p in data_pts])
sponge_arr = np.array([p['sponge_time'] for p in data_pts])
total_arr = np.array([p['total_time'] for p in data_pts])

coi_mask = coi_arr > 1e-6; sponge_mask = sponge_arr > 1e-6; total_mask = total_arr > 1e-6

ax.scatter(nf_arr[coi_mask], coi_arr[coi_mask], c='#4477AA', marker='o', s=50,
           label='CoI computation', alpha=0.8, edgecolors='black', linewidths=0.5, zorder=3)
ax.scatter(nf_arr[sponge_mask], sponge_arr[sponge_mask], c='#EE6677', marker='^', s=50,
           label='SPONGE clustering', alpha=0.8, edgecolors='black', linewidths=0.5, zorder=3)
ax.scatter(nf_arr[total_mask], total_arr[total_mask], c='#228833', marker='s', s=50,
           label='Total pipeline', alpha=0.8, edgecolors='black', linewidths=0.5, zorder=3)

exponents = {}
for arr, mask, name, color in [(coi_arr, coi_mask, 'coi', '#4477AA'),
                                (sponge_arr, sponge_mask, 'sponge', '#EE6677'),
                                (total_arr, total_mask, 'total', '#228833')]:
    v_nf = nf_arr[mask]; v_t = arr[mask]
    if len(v_nf) >= 3 and len(np.unique(v_nf)) >= 2:
        log_nf = np.log10(v_nf.astype(float)); log_t = np.log10(v_t.astype(float))
        coeffs = np.polyfit(log_nf, log_t, 1)
        exponents[name] = float(coeffs[0])
        nf_range = np.linspace(v_nf.min(), v_nf.max(), 100)
        fit_line = 10 ** np.polyval(coeffs, np.log10(nf_range))
        ax.plot(nf_range, fit_line, color=color, linestyle='--', linewidth=1.5, alpha=0.6)
        mid = len(nf_range) // 2
        ax.annotate(f'O(d$^{{{coeffs[0]:.1f}}}$)', xy=(nf_range[mid], fit_line[mid]),
                    xytext=(10, 10), textcoords='offset points', fontsize=8, color=color, fontweight='bold')

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Number of Features (d)'); ax.set_ylabel('Time (seconds)')
ax.set_title('Computational Scaling Analysis', fontweight='bold')
ax.legend(fontsize=8, frameon=True)
for p in data_pts:
    if p['n_features'] >= 50 or p['n_features'] <= 8:
        ax.annotate(p['dataset'], xy=(p['n_features'], p['total_time']),
                    xytext=(5, 5), textcoords='offset points', fontsize=6, alpha=0.7)
plt.tight_layout()
plt.show()
print(f"Power-law exponents: {exponents}")

Power-law exponents: {'coi': 1.632519588585857, 'sponge': 1.0174394679788046, 'total': 1.5380819968099608}


## Results Summary

In [15]:
# Compute summary statistics
method_avg = {}
for method in ALL_METHODS:
    vals = []
    for ds in CLASSIFICATION_DATASETS:
        key = (ds, method)
        if key in figs_data: vals.append(figs_data[key]['ba_mean'])
        elif key in base_data: vals.append(base_data[key]['ba_mean'])
    method_avg[method] = float(np.mean(vals)) if vals else 0.0

print("=" * 60)
print("EVALUATION SUMMARY: 7 Publication-Quality Figures")
print("=" * 60)
print()
print(f"Friedman chi2 = {fstat:.3f}, p = {fpval:.6f}")
print(f"Critical Difference (CD) = {cd:.3f}")
print()
print(f"Method Average Balanced Accuracy (across {len(CLASSIFICATION_DATASETS)} datasets):")
print("-" * 45)
for m in sorted(method_avg, key=lambda x: -method_avg[x]):
    bar = "#" * int(method_avg[m] * 40)
    print(f"  {METHOD_LABELS[m]:22s} {method_avg[m]:.4f}  {bar}")

print()
print("Hedges' g effect sizes (signed vs unsigned):")
print("-" * 45)
for name, g_val in hedges_g_results.items():
    direction = "<- signed" if g_val < 0 else "unsigned ->"
    print(f"  {name:25s}  g={g_val:+.3f}  ({direction})")

if exponents:
    print()
    print("Scaling exponents:")
    for name, exp_val in exponents.items():
        print(f"  {name}: O(d^{exp_val:.2f})")

print()
print("=" * 60)
print("All 7 figures generated successfully!")
print("=" * 60)

EVALUATION SUMMARY: 7 Publication-Quality Figures

Friedman chi2 = 37.095, p = 0.000004
Critical Difference (CD) = 3.969

Method Average Balanced Accuracy (across 7 datasets):
---------------------------------------------
  Random Forest          0.7765  ###############################
  EBM                    0.7703  ##############################
  FIGS (Hard-Thr.)       0.7358  #############################
  FIGS (Sgn-Spec.)       0.7357  #############################
  FIGS (Uns-Spec.)       0.7351  #############################
  FIGS (Rand-Obl.)       0.7323  #############################
  FIGS (Axis)            0.7282  #############################
  Logistic/Ridge         0.7044  ############################

Hedges' g effect sizes (signed vs unsigned):
---------------------------------------------
  adult                      g=+0.010  (unsigned ->)
  electricity                g=-0.325  (<- signed)
  credit                     g=+0.016  (unsigned ->)
  eye_movements        